# MU Editing QC and Cross-Angle Motor Unit Tracking

This notebook mirrors the latest analysis logic in `run_report_analysis_all.py`. It evaluates manual MU editing, extracts MUAPs, tracks motor units across joint angles, and validates the accepted links against alternative matches.


## Module 1 — Configuration and data structures

Imports dependencies, defines input/output paths, analysis thresholds, preprocessing settings, and shared data containers.


In [ ]:
from __future__ import annotations

import argparse
import csv
import math
import os
import re
import shutil
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

os.environ.setdefault("OMP_NUM_THREADS", "1")
warnings.filterwarnings("ignore", message="KMeans is known.*", category=UserWarning)

import h5py
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyArrowPatch
from scipy import signal
from scipy.io import loadmat
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans

from motor_unit_toolbox.props import (
    center_muaps,
    get_coefficient_of_variation,
    get_discharge_rate,
    get_muaps,
    get_number_of_spikes,
    get_pulse_to_noise_ratio,
)
from motor_unit_toolbox.muap_comp import assign_muaps_all_trials, compute_muaps_dist_sets

try:
    from emg_toolbox.prepro import highpass_filter, lowpass_filter, remove_powerline
    from emg_toolbox.tools import replace_bad_ch
except Exception:
    highpass_filter = None
    lowpass_filter = None
    remove_powerline = None
    replace_bad_ch = None


# Expected input structure:
#   <DATA_ROOT>/forearm_muedit_new/<subject>/*.mat_edited.mat
#   <DATA_ROOT>/wrist_muedit_new/<subject>/*.mat_edited.mat
# Each edited MAT file must contain `signal` (data, fs/fsamp) and `edition`
# (Distimeclean/Dischargetimes, channelmap; badchannel is optional).
# A matching raw `<recording>.mat` is optional and is used as the pre-edit source.
# e.g. Path(r"D:\zhuomian\Report_file\Report_Analysis")
DATA_ROOT = Path(r"")

# Output directory, e.g. Path(r"D:\zhuomian\Report_file\Analysis_Result_plot_table")
RESULT_ROOT = Path(r"")

LOCATION_DIRS = {
    "forearm": "forearm_muedit_new",
    "wrist": "wrist_muedit_new",
}
TASKS = ["index-middle", "index", "middle"]

EXT_FACT = 8
WIN_MS = 25
MATCH_ROA_THR = 0.30
TOL_SPIKE_MS = 1
TOL_TRAIN_MS = 40

DIST_METRIC = "nmse"
DIST_THR = 0.20
SEL_CHS = "iqr"

DO_FILTER = True
LOW_PASS_CUTOFF = 500
HIGH_PASS_CUTOFF = 20
NOTCH = 50
NOTCH_WIDTH = 1
FILTER_ORDER_HP = 2
FILTER_ORDER_LP = 2
FILTER_ORDER_NOTCH = 2
FILT_FILT = True
DO_REPLACE_BAD_CH = True


@dataclass
class GroupData:
    discharge_times: list[np.ndarray]
    ipt: np.ndarray
    fs: float
    timestamps: np.ndarray


@dataclass
class FileMeta:
    subject: str
    location: str
    task: str
    angle: int | None


## Module 2 — MATLAB data loading and spike-train preparation

Loads classic or HDF5 MATLAB files, extracts MU discharge data, parses file metadata, and builds aligned spike-train arrays.


In [ ]:
def is_hdf5_mat(path: Path) -> bool:
    try:
        with h5py.File(path, "r"):
            return True
    except OSError:
        return False


def _is_h5_ref(value) -> bool:
    return isinstance(value, h5py.h5r.Reference)


def _deref_h5(f: h5py.File, obj, depth: int = 0):
    if depth > 8:
        return np.asarray(obj).squeeze()

    if isinstance(obj, h5py.Dataset):
        arr = np.asarray(obj)
    else:
        arr = np.asarray(obj)

    if arr.dtype == object:
        out = []
        for item in arr.reshape(-1):
            if _is_h5_ref(item):
                out.append(_deref_h5(f, f[item], depth + 1))
            else:
                out.append(np.asarray(item).squeeze())
        return out
    return arr.squeeze()


def _unwrap_single_cell(value):
    while isinstance(value, list) and len(value) == 1:
        value = value[0]
    return value


def _as_discharge_list(value) -> list[np.ndarray]:
    if value is None:
        return []
    if isinstance(value, list):
        out = []
        for item in value:
            if isinstance(item, list):
                out.extend(_as_discharge_list(item))
            else:
                out.append(np.asarray(item).astype(float).squeeze())
        return out
    arr = np.asarray(value)
    if arr.dtype == object:
        return [np.asarray(x).astype(float).squeeze() for x in arr.reshape(-1)]
    if arr.ndim == 0:
        return [arr.astype(float).reshape(1)]
    return [arr.astype(float).squeeze()]


def _orient_samples_by_units(arr: np.ndarray, n_units_hint: int | None = None) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.ndim == 0:
        arr = arr.reshape(1, 1)
    if arr.ndim == 1:
        arr = arr[:, None]
    if arr.ndim != 2:
        arr = np.squeeze(arr)
        if arr.ndim == 1:
            arr = arr[:, None]

    if n_units_hint is not None and n_units_hint > 0:
        if arr.shape[1] == n_units_hint:
            return arr
        if arr.shape[0] == n_units_hint:
            return arr.T

    if arr.shape[0] < arr.shape[1] and arr.shape[0] <= 512:
        return arr.T
    return arr


def _get_mat_struct_field(obj, names: Iterable[str]):
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    return None


def load_group_any(path: Path, group_name: str, prefer_clean: bool = False) -> GroupData:
    if is_hdf5_mat(path):
        with h5py.File(path, "r") as f:
            if group_name not in f:
                raise KeyError(f"{group_name} group not found in {path.name}")
            grp = f[group_name]

            dt_raw = None
            if "Dischargetimes" in grp:
                dt_raw = _deref_h5(f, grp["Dischargetimes"])
            elif "Distimeclean" in grp:
                dt_raw = _deref_h5(f, grp["Distimeclean"])
            discharge_times = _as_discharge_list(dt_raw)

            ipt_key_candidates = (
                ["Pulsetrainclean", "IPTs", "ipts", "Pulsetrain"]
                if prefer_clean
                else ["IPTs", "ipts", "Pulsetrain", "Pulsetrainclean"]
            )
            ipt = None
            for key in ipt_key_candidates:
                if key in grp:
                    ipt = _unwrap_single_cell(_deref_h5(f, grp[key]))
                    break
            if ipt is None:
                raise KeyError(f"No pulse train found in {group_name} of {path.name}")

            fs = None
            for key in ["fsamp", "fs", "Fs", "FS"]:
                if key in grp:
                    fs = float(np.asarray(grp[key]).squeeze())
                    break
            if fs is None and "signal" in f:
                sig = f["signal"]
                for key in ["fsamp", "fs", "Fs", "FS"]:
                    if key in sig:
                        fs = float(np.asarray(sig[key]).squeeze())
                        break
            if fs is None:
                raise KeyError(f"No sampling rate found in {path.name}")

            ipt = _orient_samples_by_units(np.asarray(ipt, dtype=float), len(discharge_times))
            timestamps = np.arange(ipt.shape[0], dtype=float) / fs
            return GroupData(discharge_times, ipt, fs, timestamps)

    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    if group_name not in mat:
        raise KeyError(f"{group_name} group not found in {path.name}")
    grp = mat[group_name]

    dt_raw = _get_mat_struct_field(grp, ["Dischargetimes", "Distimeclean"])
    discharge_times = _as_discharge_list(dt_raw)

    ipt_raw = None
    ipt_keys = (
        ["Pulsetrainclean", "IPTs", "ipts", "Ipts", "ipt", "IPT", "Pulsetrain"]
        if prefer_clean
        else ["IPTs", "ipts", "Ipts", "ipt", "IPT", "Pulsetrain", "Pulsetrainclean"]
    )
    for key in ipt_keys:
        if hasattr(grp, key):
            ipt_raw = getattr(grp, key)
            break
    if ipt_raw is None:
        raise KeyError(f"No pulse train found in {group_name} of {path.name}")
    if isinstance(ipt_raw, np.ndarray) and ipt_raw.dtype == object:
        ipt_raw = ipt_raw.reshape(-1)[0]

    fs = _get_mat_struct_field(grp, ["fsamp", "fs", "Fs", "FS"])
    if fs is None and "signal" in mat:
        fs = _get_mat_struct_field(mat["signal"], ["fsamp", "fs", "Fs", "FS"])
    if fs is None:
        raise KeyError(f"No sampling rate found in {path.name}")
    fs = float(np.asarray(fs).squeeze())

    ipt = _orient_samples_by_units(np.asarray(ipt_raw, dtype=float), len(discharge_times))
    timestamps = np.arange(ipt.shape[0], dtype=float) / fs
    return GroupData(discharge_times, ipt, fs, timestamps)


def parse_file_meta(path: Path) -> FileMeta:
    subject = path.parent.name
    location = ""
    for loc, dirname in LOCATION_DIRS.items():
        if dirname.lower() in [part.lower() for part in path.parts]:
            location = loc
            break
    m_task = re.search(r"_flx_1_([^_]+)_15MVC_", path.name)
    m_angle = re.search(r"_decomp_(-?\d+)deg", path.name)
    task = m_task.group(1) if m_task else ""
    angle = int(m_angle.group(1)) if m_angle else None
    return FileMeta(subject=subject, location=location, task=task, angle=angle)


def raw_before_path(edited_path: Path) -> Path | None:
    candidate = edited_path.with_name(edited_path.name.replace(".mat_edited.mat", ".mat"))
    return candidate if candidate.exists() else None


def build_spike_train(discharge_times: list[np.ndarray], n_samples: int, n_units: int | None = None) -> np.ndarray:
    if n_units is None:
        n_units = len(discharge_times)
    st = np.zeros((n_samples, n_units), dtype=bool)
    for unit, times in enumerate(discharge_times[:n_units]):
        if times is None or np.size(times) == 0:
            continue
        idx = np.asarray(times).astype(float).ravel()
        idx = idx[np.isfinite(idx)].astype(int)
        idx = idx[(idx >= 0) & (idx < n_samples)]
        if idx.size:
            st[idx, unit] = True
    return st


def pad_or_trim_ipts(ipt: np.ndarray, n_samples: int, n_units: int) -> np.ndarray:
    out = np.full((n_samples, n_units), np.nan, dtype=float)
    if ipt.size == 0:
        return out
    rows = min(n_samples, ipt.shape[0])
    cols = min(n_units, ipt.shape[1])
    out[:rows, :cols] = ipt[:rows, :cols]
    return out


## Module 3 — MU quality metrics and before/after matching

Computes discharge rate, CoV, PNR, SIL, and spike counts, then aligns and matches MUs with full-RoA Hungarian assignment.


In [ ]:
def compute_basic_metrics(group: GroupData, n_samples: int) -> dict[str, np.ndarray]:
    n_units = max(len(group.discharge_times), group.ipt.shape[1])
    st = build_spike_train(group.discharge_times, n_samples, n_units=n_units)
    ipt = pad_or_trim_ipts(group.ipt, n_samples, n_units)
    ts = np.arange(n_samples, dtype=float) / group.fs

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        dr = get_discharge_rate(st, ts)
        cov = get_coefficient_of_variation(st, ts)
        nsp = get_number_of_spikes(st)

    pnr = np.full(n_units, np.nan)
    finite_ipt_cols = np.where(~np.isnan(ipt).all(axis=0))[0]
    if finite_ipt_cols.size:
        common = int(finite_ipt_cols.max()) + 1
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                pnr[:common] = get_pulse_to_noise_ratio(
                    st[:, :common],
                    np.nan_to_num(ipt[:, :common], nan=0.0),
                    ext_fact=EXT_FACT,
                )
        except Exception:
            for unit in finite_ipt_cols:
                try:
                    pnr[unit] = get_pulse_to_noise_ratio(
                        st[:, [unit]],
                        np.nan_to_num(ipt[:, [unit]], nan=0.0),
                        ext_fact=EXT_FACT,
                    )[0]
                except Exception:
                    pnr[unit] = np.nan

    sil = np.full(n_units, np.nan)
    n_peaks = np.zeros(n_units, dtype=int)
    for unit in range(min(n_units, group.ipt.shape[1])):
        sil[unit], n_peaks[unit] = calc_sil_matlab_like(group.ipt[:n_samples, unit], group.fs)

    return {
        "st": st,
        "ipt": ipt,
        "discharge_rate": dr,
        "cov": cov,
        "pnr": pnr,
        "sil": sil,
        "sil_n_peaks": n_peaks,
        "n_spikes": nsp.astype(int),
    }


def calc_sil_matlab_like(pulse_train: np.ndarray, fs: float) -> tuple[float, int]:
    """MATLAB calcSIL logic applied to an already computed MU pulse train."""
    x = np.asarray(pulse_train, dtype=float).ravel()
    if x.size == 0 or not np.any(np.isfinite(x)):
        return np.nan, 0
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    min_dist = max(1, int(round(fs * 0.02)))
    peaks, _ = signal.find_peaks(x, distance=min_dist)
    if peaks.size <= 1:
        return 0.0, int(peaks.size)

    peak_values = x[peaks]
    top_n = min(10, peak_values.size)
    scale = float(np.mean(np.sort(peak_values)[-top_n:]))
    if np.isfinite(scale) and scale != 0:
        x = x / scale
        peak_values = x[peaks]

    if np.unique(peak_values).size < 2:
        return 0.0, int(peaks.size)

    try:
        km = KMeans(n_clusters=2, n_init=10, random_state=0)
        labels = km.fit_predict(peak_values.reshape(-1, 1))
        centers = km.cluster_centers_.reshape(-1)
    except Exception:
        return np.nan, int(peaks.size)

    spike_cluster = int(np.argmax(centers))
    other_cluster = 1 - spike_cluster
    selected = labels == spike_cluster
    if not np.any(selected):
        return 0.0, int(peaks.size)

    vals = peak_values[selected]
    within = float(np.sum((vals - centers[spike_cluster]) ** 2))
    between = float(np.sum((vals - centers[other_cluster]) ** 2))
    denom = max(within, between)
    if denom == 0 or not np.isfinite(denom):
        return 0.0, int(peaks.size)
    return float((between - within) / denom), int(peaks.size)


def compare_spike_trains(ref: np.ndarray, test: np.ndarray, fs: float) -> tuple[float, int, int, int, int]:
    ref = np.asarray(ref).astype(bool)
    test = np.asarray(test).astype(bool)
    tol_spike = max(1, int(round(TOL_SPIKE_MS / 1000 * fs)))
    tol_train = max(0, int(round(TOL_TRAIN_MS / 1000 * fs)))

    if not np.any(ref) and not np.any(test):
        return 0.0, 0, 0, 0, 0
    if not np.any(ref):
        return 0.0, 0, 0, 0, int(np.sum(test))
    if not np.any(test):
        return 0.0, 0, 0, int(np.sum(ref)), 0

    ref_conv = np.convolve(ref.astype(float), np.ones(tol_spike), mode="same")
    test_conv = np.convolve(test.astype(float), np.ones(tol_spike), mode="same")
    corr = signal.correlate(ref_conv, test_conv, mode="full")
    lags = signal.correlation_lags(ref_conv.size, test_conv.size, mode="full")
    mask = np.abs(lags) <= tol_train
    if np.any(mask):
        corr = corr[mask]
        lags = lags[mask]
    lag = int(lags[np.argmax(np.abs(corr))]) if corr.size else 0

    firings_ref = np.flatnonzero(ref)
    firings_test = np.flatnonzero(test) + lag
    common = 0
    ref_only = 0
    remaining_test = firings_test.astype(int).copy()

    for firing in firings_ref:
        if remaining_test.size == 0:
            ref_only += 1
            continue
        diffs = np.abs(remaining_test - firing)
        if np.any(diffs <= tol_spike):
            common += 1
            remaining_test = np.delete(remaining_test, int(np.argmin(diffs)))
        else:
            ref_only += 1

    test_only = int(remaining_test.size)
    denom = common + ref_only + test_only
    roa = float(common / denom) if denom else 0.0
    return roa, lag, int(common), int(ref_only), int(test_only)


def build_roa_matrices(st_before: np.ndarray, st_after: np.ndarray, fs: float):
    n_before = st_before.shape[1]
    n_after = st_after.shape[1]
    roa = np.zeros((n_before, n_after), dtype=float)
    lag = np.zeros((n_before, n_after), dtype=int)
    common = np.zeros((n_before, n_after), dtype=int)
    deleted = np.zeros((n_before, n_after), dtype=int)
    added = np.zeros((n_before, n_after), dtype=int)

    for i in range(n_before):
        for j in range(n_after):
            stats = compare_spike_trains(st_before[:, i], st_after[:, j], fs)
            roa[i, j], lag[i, j], common[i, j], deleted[i, j], added[i, j] = stats
    return roa, lag, common, deleted, added


def hungarian_match(roa: np.ndarray, min_roa: float) -> dict[int, int]:
    if roa.size == 0 or roa.shape[0] == 0 or roa.shape[1] == 0:
        return {}
    rows, cols = linear_sum_assignment(-roa)
    matches = {}
    for row, col in zip(rows, cols):
        if np.isfinite(roa[row, col]) and roa[row, col] >= min_roa:
            matches[int(row)] = int(col)
    return matches


## Module 4 — Per-file manual-editing comparison

Classifies matched, deleted, and newly added MUs and records before/after metrics, deltas, and spike-level editing changes.


In [ ]:
def _val(arr: np.ndarray, idx: int, default=np.nan):
    if idx is None or idx < 0 or idx >= len(arr):
        return default
    value = arr[idx]
    if isinstance(value, np.generic):
        return value.item()
    return value


def compute_single_file_metrics(edited_path: Path, min_roa: float) -> tuple[list[dict], list[dict], dict]:
    meta = parse_file_meta(edited_path)
    before_path = raw_before_path(edited_path)
    before_group_path = before_path if before_path is not None else edited_path
    before_group_name = "signal"

    before = load_group_any(before_group_path, before_group_name, prefer_clean=False)
    after = load_group_any(edited_path, "edition", prefer_clean=True)

    n_samples = min(before.ipt.shape[0], after.ipt.shape[0])
    before_metrics = compute_basic_metrics(before, n_samples)
    after_metrics = compute_basic_metrics(after, n_samples)

    st_before = before_metrics["st"]
    st_after = after_metrics["st"]
    roa, lag, common, deleted, added = build_roa_matrices(st_before, st_after, before.fs)
    matches = hungarian_match(roa, min_roa)
    matched_after = set(matches.values())

    base = {
        "subject": meta.subject,
        "location": meta.location,
        "task": meta.task,
        "angle_deg": meta.angle,
        "file_stem": edited_path.name,
        "before_file": str(before_group_path),
        "after_file": str(edited_path),
        "before_source": "raw_mat" if before_path is not None else "edited_signal_group",
        "match_method": "hungarian_full_roa",
        "match_roa_threshold": min_roa,
    }

    rows = []
    match_rows = []
    n_before = st_before.shape[1]
    n_after = st_after.shape[1]

    for before_idx in range(n_before):
        after_idx = matches.get(before_idx)
        best_after_idx = int(np.nanargmax(roa[before_idx])) if n_after else None
        best_roa = float(roa[before_idx, best_after_idx]) if best_after_idx is not None else np.nan

        if after_idx is None:
            row = make_metric_row(
                base,
                before_metrics,
                after_metrics,
                before_idx,
                None,
                np.nan,
                np.nan,
                0,
                int(_val(before_metrics["n_spikes"], before_idx, 0)),
                0,
                deleted_mu=True,
                new_mu=False,
                matched_mu=False,
                best_after_unit_idx=best_after_idx,
                best_candidate_roa=best_roa,
            )
            rows.append(row)
            match_rows.append(
                make_match_row(base, before_idx, None, np.nan, np.nan, 0, row["n_deleted_spikes"], 0, "deleted")
            )
            continue

        row = make_metric_row(
            base,
            before_metrics,
            after_metrics,
            before_idx,
            after_idx,
            float(roa[before_idx, after_idx]),
            int(lag[before_idx, after_idx]),
            int(common[before_idx, after_idx]),
            int(deleted[before_idx, after_idx]),
            int(added[before_idx, after_idx]),
            deleted_mu=False,
            new_mu=False,
            matched_mu=True,
            best_after_unit_idx=best_after_idx,
            best_candidate_roa=best_roa,
        )
        rows.append(row)
        match_rows.append(
            make_match_row(
                base,
                before_idx,
                after_idx,
                row["RoA"],
                row["alignment_lag"],
                row["n_common_spikes"],
                row["n_deleted_spikes"],
                row["n_added_spikes"],
                "matched",
            )
        )

    for after_idx in range(n_after):
        if after_idx in matched_after:
            continue
        row = make_metric_row(
            base,
            before_metrics,
            after_metrics,
            None,
            after_idx,
            np.nan,
            np.nan,
            0,
            0,
            int(_val(after_metrics["n_spikes"], after_idx, 0)),
            deleted_mu=False,
            new_mu=True,
            matched_mu=False,
            best_after_unit_idx=after_idx,
            best_candidate_roa=np.nan,
        )
        rows.append(row)
        match_rows.append(
            make_match_row(base, None, after_idx, np.nan, np.nan, 0, 0, row["n_added_spikes"], "new_after")
        )

    summary = {
        **base,
        "n_before_mu": n_before,
        "n_after_mu": n_after,
        "n_matched_mu": int(sum(r["matched_mu"] for r in rows)),
        "n_deleted_mu": int(sum(r["deleted_mu"] for r in rows)),
        "n_new_after_mu": int(sum(r["new_mu"] for r in rows)),
        "mean_roa_matched": float(np.nanmean([r["RoA"] for r in rows if r["matched_mu"]]))
        if any(r["matched_mu"] for r in rows)
        else np.nan,
        "total_added_spikes": int(np.nansum([r["n_added_spikes"] for r in rows])),
        "total_deleted_spikes": int(np.nansum([r["n_deleted_spikes"] for r in rows])),
    }
    return rows, match_rows, summary


def make_metric_row(
    base: dict,
    before_metrics: dict[str, np.ndarray],
    after_metrics: dict[str, np.ndarray],
    before_idx: int | None,
    after_idx: int | None,
    roa: float,
    alignment_lag: float,
    n_common: int,
    n_deleted: int,
    n_added: int,
    deleted_mu: bool,
    new_mu: bool,
    matched_mu: bool,
    best_after_unit_idx: int | None,
    best_candidate_roa: float,
) -> dict:
    def metric(prefix: str, metrics: dict[str, np.ndarray], idx: int | None, key: str, default=np.nan):
        return _val(metrics[key], idx, default) if idx is not None else default

    row = {
        **base,
        "unit_idx_before": np.nan if before_idx is None else int(before_idx),
        "unit_idx_after": np.nan if after_idx is None else int(after_idx),
        "unit_idx": np.nan if before_idx is None else int(before_idx),
        "matched_mu": bool(matched_mu),
        "deleted_mu": bool(deleted_mu),
        "new_mu": bool(new_mu),
        "best_after_unit_idx": np.nan if best_after_unit_idx is None else int(best_after_unit_idx),
        "best_candidate_roa": best_candidate_roa,
        "RoA": roa,
        "alignment_lag": alignment_lag,
        "n_common_spikes": int(n_common),
        "n_added_spikes": int(n_added),
        "n_deleted_spikes": int(n_deleted),
        "discharge_rate_before": metric("before", before_metrics, before_idx, "discharge_rate"),
        "discharge_rate_after": metric("after", after_metrics, after_idx, "discharge_rate"),
        "cov_before": metric("before", before_metrics, before_idx, "cov"),
        "cov_after": metric("after", after_metrics, after_idx, "cov"),
        "pnr_before": metric("before", before_metrics, before_idx, "pnr"),
        "pnr_after": metric("after", after_metrics, after_idx, "pnr"),
        "sil_before": metric("before", before_metrics, before_idx, "sil"),
        "sil_after": metric("after", after_metrics, after_idx, "sil"),
        "sil_n_peaks_before": metric("before", before_metrics, before_idx, "sil_n_peaks", 0),
        "sil_n_peaks_after": metric("after", after_metrics, after_idx, "sil_n_peaks", 0),
        "n_spikes_before": int(metric("before", before_metrics, before_idx, "n_spikes", 0)),
        "n_spikes_after": int(metric("after", after_metrics, after_idx, "n_spikes", 0)),
    }
    for key in ["discharge_rate", "cov", "pnr", "sil"]:
        row[f"{key}_delta"] = row[f"{key}_after"] - row[f"{key}_before"]
    row["n_spikes_delta"] = row["n_spikes_after"] - row["n_spikes_before"]
    return row


def make_match_row(
    base: dict,
    before_idx: int | None,
    after_idx: int | None,
    roa: float,
    lag: float,
    n_common: int,
    n_deleted: int,
    n_added: int,
    status: str,
) -> dict:
    return {
        **base,
        "unit_idx_before": np.nan if before_idx is None else int(before_idx),
        "unit_idx_after": np.nan if after_idx is None else int(after_idx),
        "match_status": status,
        "RoA": roa,
        "alignment_lag": lag,
        "n_common_spikes": int(n_common),
        "n_deleted_spikes": int(n_deleted),
        "n_added_spikes": int(n_added),
    }


## Module 5 — Batch QC tables and distribution plots

Runs the per-file analysis for every subject and location, then exports subject-level and global CSV, Excel, and histogram outputs.


In [ ]:
def discover_subject_location_dirs(data_root: Path) -> list[tuple[str, str, Path]]:
    out = []
    for location, dirname in LOCATION_DIRS.items():
        loc_dir = data_root / dirname
        if not loc_dir.exists():
            continue
        for sub_dir in sorted(p for p in loc_dir.iterdir() if p.is_dir()):
            out.append((sub_dir.name, location, sub_dir))
    return out


def save_dataframe(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def save_excel(path: Path, sheets: dict[str, pd.DataFrame]):
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for name, df in sheets.items():
                df.to_excel(writer, sheet_name=name[:31], index=False)
    except Exception as exc:
        print(f"  !! Could not save Excel {path}: {exc}")


def plot_hist(ax, values, title, xlabel, color):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        ax.text(0.5, 0.5, "No finite data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        return
    bins = min(40, max(8, int(np.sqrt(arr.size) * 2)))
    ax.hist(arr, bins=bins, color=color, edgecolor="black", linewidth=0.5, alpha=0.82)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")


def save_distribution_plots(df: pd.DataFrame, fig_dir: Path, title_prefix: str):
    fig_dir.mkdir(parents=True, exist_ok=True)
    metrics = [
        ("discharge_rate_delta", "Delta discharge rate", "Hz", "#f27c62"),
        ("cov_delta", "Delta CoV", "%", "#8fbf70"),
        ("pnr_delta", "Delta PNR", "dB", "#8e77bd"),
        ("sil_delta", "Delta silhouette", "a.u.", "#6fa8dc"),
        ("n_added_spikes", "Added spikes per MU", "Spikes", "#f4a261"),
        ("n_deleted_spikes", "Deleted spikes per MU", "Spikes", "#d76f6f"),
        ("RoA", "Rate of agreement", "RoA", "#5b8cc0"),
    ]

    fig, axes = plt.subplots(3, 3, figsize=(13, 10))
    axes_flat = axes.ravel()
    for ax, (col, title, xlabel, color) in zip(axes_flat, metrics):
        plot_hist(ax, df[col] if col in df else [], title, xlabel, color)
    for ax in axes_flat[len(metrics) :]:
        ax.axis("off")
    fig.suptitle(title_prefix, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(fig_dir / "all_metric_distributions.png", dpi=250)
    plt.close(fig)

    for col, title, xlabel, color in metrics:
        fig, ax = plt.subplots(figsize=(5.5, 4))
        plot_hist(ax, df[col] if col in df else [], title, xlabel, color)
        fig.tight_layout()
        fig.savefig(fig_dir / f"{col}_distribution.png", dpi=250)
        plt.close(fig)


def run_metrics(data_root: Path, out_root: Path, min_roa: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    all_rows = []
    all_match_rows = []
    all_summaries = []
    errors = []

    for subject, location, sub_dir in discover_subject_location_dirs(data_root):
        print(f"\n[metrics] {subject} {location}")
        edited_files = sorted(p for p in sub_dir.glob("*.mat_edited.mat") if "dataZero" not in p.name)
        rows = []
        match_rows = []
        summaries = []
        for idx, path in enumerate(edited_files, start=1):
            print(f"  [{idx}/{len(edited_files)}] {path.name}")
            try:
                r, m, s = compute_single_file_metrics(path, min_roa=min_roa)
                rows.extend(r)
                match_rows.extend(m)
                summaries.append(s)
            except Exception as exc:
                err = {
                    "stage": "metrics",
                    "subject": subject,
                    "location": location,
                    "file": str(path),
                    "error": repr(exc),
                }
                errors.append(err)
                print(f"    !! ERROR: {exc}")

        df = pd.DataFrame(rows)
        df_match = pd.DataFrame(match_rows)
        df_summary = pd.DataFrame(summaries)
        out_dir = out_root / subject / location
        table_dir = out_dir / "tables"
        fig_dir = out_dir / "figures" / "distributions"

        if not df.empty:
            save_dataframe(df, table_dir / "per_mu_analysis.csv")
            save_dataframe(df_match, table_dir / "before_after_mu_matching.csv")
            save_dataframe(df_summary, table_dir / "per_file_summary.csv")
            save_excel(
                table_dir / "analysis_tables.xlsx",
                {
                    "per_mu_analysis": df,
                    "before_after_matching": df_match,
                    "per_file_summary": df_summary,
                },
            )
            save_distribution_plots(df, fig_dir, f"{subject} {location}: before/after MU analysis")

        all_rows.extend(rows)
        all_match_rows.extend(match_rows)
        all_summaries.extend(summaries)

    df_all = pd.DataFrame(all_rows)
    df_match_all = pd.DataFrame(all_match_rows)
    df_summary_all = pd.DataFrame(all_summaries)
    global_table_dir = out_root / "_global" / "tables"
    global_fig_dir = out_root / "_global" / "figures" / "distributions"
    if not df_all.empty:
        save_dataframe(df_all, global_table_dir / "all_per_mu_analysis.csv")
        save_dataframe(df_match_all, global_table_dir / "all_before_after_mu_matching.csv")
        save_dataframe(df_summary_all, global_table_dir / "all_per_file_summary.csv")
        save_excel(
            global_table_dir / "all_analysis_tables.xlsx",
            {
                "all_per_mu_analysis": df_all,
                "all_before_after_matching": df_match_all,
                "all_per_file_summary": df_summary_all,
            },
        )
        save_distribution_plots(df_all, global_fig_dir, "All subjects and locations: before/after MU analysis")

    if errors:
        save_dataframe(pd.DataFrame(errors), out_root / "_global" / "tables" / "errors.csv")
    return df_all, df_match_all, df_summary_all


## Module 6 — EMG preprocessing and MUAP extraction

Replaces bad channels, filters EMG, converts cleaned discharges to spike trains, extracts centered 25-ms MUAPs, and saves compressed NPZ files.


In [ ]:
def _deref_cell_h5_for_muap(f: h5py.File, ds) -> list[np.ndarray]:
    arr = np.asarray(ds)
    if arr.size == 1 and _is_h5_ref(arr.reshape(-1)[0]):
        arr = np.asarray(f[arr.reshape(-1)[0]])
    out = []
    for item in arr.reshape(-1):
        if _is_h5_ref(item):
            out.append(np.asarray(f[item]).squeeze())
        else:
            out.append(np.asarray(item).squeeze())
    return out


def load_muedit_for_muap(path: Path):
    if is_hdf5_mat(path):
        with h5py.File(path, "r") as f:
            fs = int(np.asarray(f["signal/fs" if "fs" in f["signal"] else "signal/fsamp"]).squeeze())
            data = np.asarray(f["signal/data"])
            if data.ndim == 2 and data.shape[0] < data.shape[1] and data.shape[0] in (128, 192, 256):
                data = data.T
            channelmap = np.asarray(f["edition/channelmap"]).astype(int)
            if "Distimeclean" in f["edition"]:
                firing_list = _deref_cell_h5_for_muap(f, f["edition/Distimeclean"])
            else:
                firing_list = _deref_cell_h5_for_muap(f, f["edition/Dischargetimes"])
            bad_ch_1based = (
                np.asarray(f["edition/badchannel"]).astype(int).squeeze()
                if "badchannel" in f["edition"]
                else np.asarray([], dtype=int)
            )
        return fs, data, channelmap, firing_list, bad_ch_1based

    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    fs = int(np.asarray(_get_mat_struct_field(mat["signal"], ["fs", "fsamp"])).squeeze())
    data = np.asarray(mat["signal"].data)
    if data.ndim == 2 and data.shape[0] < data.shape[1] and data.shape[0] in (128, 192, 256):
        data = data.T
    channelmap = np.asarray(mat["edition"].channelmap).astype(int)
    dc = _get_mat_struct_field(mat["edition"], ["Distimeclean", "Dischargetimes"])
    firing_list = _as_discharge_list(dc)
    bad_ch_1based = (
        np.asarray(mat["edition"].badchannel).astype(int).squeeze()
        if hasattr(mat["edition"], "badchannel")
        else np.asarray([], dtype=int)
    )
    return fs, data, channelmap, firing_list, bad_ch_1based


def _to_0based_channels(ch_list_1based, n_ch: int, ch_map_1based: np.ndarray) -> np.ndarray:
    if ch_list_1based is None:
        return np.asarray([], dtype=int)
    ch = np.asarray(ch_list_1based).astype(int).reshape(-1)
    ch0 = ch - 1
    ch0 = ch0[(ch0 >= 0) & (ch0 < n_ch)]
    ch_map0 = (np.asarray(ch_map_1based).astype(int) - 1).reshape(-1)
    ch0 = ch0[np.isin(ch0, ch_map0)]
    return np.unique(ch0)


def preprocess_emg(data: np.ndarray, fs: int, channelmap_1based: np.ndarray, bad_ch_1based: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    data = data.astype(np.float32, copy=False)
    ch_map0 = channelmap_1based.astype(int) - 1
    bad_ch0 = _to_0based_channels(bad_ch_1based, n_ch=data.shape[1], ch_map_1based=channelmap_1based)

    if DO_REPLACE_BAD_CH and replace_bad_ch is not None and bad_ch0.size > 0:
        data = replace_bad_ch(data, bad_ch0.tolist(), ch_map0)

    if DO_FILTER:
        if highpass_filter is None or lowpass_filter is None or remove_powerline is None:
            raise RuntimeError("emg_toolbox preprocessing functions are not available")
        data = highpass_filter(data, fs=fs, cutoff=HIGH_PASS_CUTOFF, order=FILTER_ORDER_HP, filtfilt=FILT_FILT)
        data = lowpass_filter(data, fs=fs, cutoff=LOW_PASS_CUTOFF, order=FILTER_ORDER_LP, filtfilt=FILT_FILT)
        data = remove_powerline(data, fs=fs, cutoff=NOTCH, width=NOTCH_WIDTH, order=FILTER_ORDER_NOTCH, filtfilt=FILT_FILT)

    return data, bad_ch0


def build_emg_ch_array(data: np.ndarray, channelmap_1based: np.ndarray) -> np.ndarray:
    rows, cols = channelmap_1based.shape
    idx = (channelmap_1based.astype(int) - 1).reshape(-1)
    selected = data[:, idx]
    return selected.T.reshape(rows, cols, data.shape[0])


def distimeclean_to_spike_trains(firing_list: list[np.ndarray], n_samples: int, fs: int, win_ms: int) -> tuple[np.ndarray, np.ndarray]:
    n_units = len(firing_list)
    spike_trains = np.zeros((n_samples, n_units), dtype=bool)
    n_used = np.zeros(n_units, dtype=int)
    half_win = round(win_ms / 2 / 1000 * fs)
    duration_s = n_samples / fs

    for unit, firings in enumerate(firing_list):
        if firings is None or np.size(firings) == 0:
            continue
        firings = np.asarray(firings).astype(float).squeeze()
        firings = firings[np.isfinite(firings)]
        if firings.size == 0:
            continue
        max_firing = np.nanmax(firings)
        if np.isfinite(max_firing) and max_firing < duration_s * 1.2:
            idx = np.round(firings * fs).astype(int)
        else:
            idx = firings.astype(int) - 1
        idx = idx[(idx - half_win >= 0) & (idx + half_win <= n_samples - 1)]
        if idx.size:
            spike_trains[idx, unit] = True
            n_used[unit] = int(idx.size)
    return spike_trains, n_used


def compute_muap_npz_for_dir(sub_dir: Path, out_dir: Path) -> pd.DataFrame:
    muap_dir = out_dir / "muap_npz"
    muap_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    files = sorted(p for p in sub_dir.glob("*.mat_edited.mat") if "dataZero" not in p.name)
    for idx, path in enumerate(files, start=1):
        print(f"  [muap {idx}/{len(files)}] {path.name}")
        try:
            fs, data, channelmap, firing_list, bad_ch_1based = load_muedit_for_muap(path)
            if np.all(data == 0):
                print("    !! signal.data is all zeros, skipped")
                continue
            data, bad_ch0 = preprocess_emg(data, fs, channelmap, bad_ch_1based)
            emg = build_emg_ch_array(data, channelmap)
            spike_trains, n_used = distimeclean_to_spike_trains(firing_list, data.shape[0], fs, WIN_MS)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                muaps = get_muaps(spike_trains, emg, fs=fs, win_ms=WIN_MS).astype(np.float32)
                muaps = center_muaps(muaps).astype(np.float32)

            meta = parse_file_meta(path)
            out_file = muap_dir / (path.stem + f"_muap_win{WIN_MS}ms.npz")
            np.savez_compressed(
                out_file,
                muaps=muaps,
                fs=fs,
                win_ms=WIN_MS,
                channelmap=channelmap,
                n_used=n_used,
                bad_ch_1based=bad_ch_1based,
                bad_ch_0based=bad_ch0,
                prepro_hp=HIGH_PASS_CUTOFF,
                prepro_lp=LOW_PASS_CUTOFF,
                prepro_notch=NOTCH,
                prepro_notch_width=NOTCH_WIDTH,
                src=str(path),
                subject=meta.subject,
                location=meta.location,
                task=meta.task,
                angle=-9999 if meta.angle is None else meta.angle,
            )
            rows.append(
                {
                    "file": path.name,
                    "task": meta.task,
                    "angle_deg": meta.angle,
                    "fs": fs,
                    "n_samples": data.shape[0],
                    "n_ch": data.shape[1],
                    "n_units": len(firing_list),
                    "mean_spikes_used": float(np.nanmean(n_used)) if len(n_used) else np.nan,
                    "n_bad_ch": int(len(bad_ch0)),
                    "out_npz": out_file.name,
                }
            )
        except Exception as exc:
            print(f"    !! MUAP failed: {exc}")
            rows.append({"file": path.name, "error": repr(exc)})

    df = pd.DataFrame(rows)
    if not df.empty:
        save_dataframe(df, out_dir / "tables" / "muap_summary.csv")
    return df


## Module 7 — Tracking-arrow visualization

Plots every MU at each angle, connects units in the same accepted cross-angle chain, and saves both PNG and SVG versions.


In [ ]:
def plot_tracking_arrows(
    tracks: pd.DataFrame,
    angles: list[int],
    n_units_by_angle: dict[int, int],
    out_path: Path,
    title: str,
):
    if tracks.empty or not angles:
        return

    max_units = max(n_units_by_angle.values()) if n_units_by_angle else 1
    fig_w = max(6.0, 1.35 * len(angles) + 2.0)
    fig_h = max(4.5, 0.28 * max_units + 2.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    x_pos = {angle: i for i, angle in enumerate(angles)}
    cmap = plt.get_cmap("Blues")
    angle_colors = {angle: cmap(0.35 + 0.5 * i / max(1, len(angles) - 1)) for i, angle in enumerate(angles)}

    def y_for(angle: int, unit: int) -> float:
        n = n_units_by_angle.get(angle, max_units)
        return float(n - 1 - unit)

    for angle in angles:
        x = x_pos[angle]
        n_units = n_units_by_angle.get(angle, 0)
        ys = [y_for(angle, unit) for unit in range(n_units)]
        xs = [x] * n_units
        ax.scatter(xs, ys, s=58, color=angle_colors[angle], edgecolor="#4d5f75", linewidth=0.7, zorder=3)

    for _, row in tracks.iterrows():
        present = []
        for angle in angles:
            col = f"{angle}deg"
            if col not in row or pd.isna(row[col]) or row[col] == "":
                continue
            try:
                unit = int(row[col])
            except Exception:
                continue
            present.append((angle, unit))
        for (a1, u1), (a2, u2) in zip(present[:-1], present[1:]):
            x1, y1 = x_pos[a1], y_for(a1, u1)
            x2, y2 = x_pos[a2], y_for(a2, u2)
            arrow = FancyArrowPatch(
                (x1 + 0.08, y1),
                (x2 - 0.08, y2),
                arrowstyle="-|>",
                mutation_scale=9,
                linewidth=0.95,
                color="#2f3136",
                alpha=0.42,
                shrinkA=4,
                shrinkB=4,
                zorder=2,
            )
            ax.add_patch(arrow)

    ax.set_xticks([x_pos[a] for a in angles])
    ax.set_xticklabels([f"{a}°" for a in angles], fontsize=10)
    ax.set_xlim(-0.55, len(angles) - 0.45)
    ax.set_ylim(-1.0, max_units + 0.8)
    ax.set_yticks([])
    ax.set_title(title, fontsize=12)
    ax.spines[["left", "right", "top"]].set_visible(False)
    ax.spines["bottom"].set_color("#d0d0d0")
    ax.tick_params(axis="x", length=0)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=300)
    fig.savefig(out_path.with_suffix(".svg"))
    plt.close(fig)


## Module 8 — Cross-angle MU tracking

Uses grid-search assignment with NMSE and IQR-selected channels, keeps chains spanning at least two angles, and exports track/link tables and arrow plots.


In [ ]:
def run_tracking_for_dir(out_dir: Path, subject: str, location: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    muap_dir = out_dir / "muap_npz"
    files = sorted(muap_dir.glob("*.npz"))
    all_tracks = []
    all_link_rows = []
    all_validation_f = []
    all_validation_g = []

    for task_name in TASKS:
        items = []
        for f in files:
            task, angle = parse_task_angle_from_name(f.name)
            if task == task_name and angle is not None:
                items.append((angle, f))
        items.sort(key=lambda x: x[0])
        if len(items) < 2:
            print(f"  [tracking {task_name}] less than 2 angle files, skipped")
            continue
        angles = [a for a, _ in items]
        zero_trial_idx = angles.index(0) if 0 in angles else None

        muaps_list = []
        trial_labels = []
        orig_unit_idx = []
        n_units_by_angle = {}

        for trial_idx, (angle, f) in enumerate(items):
            z = np.load(f, allow_pickle=True)
            mu = z["muaps"]
            if mu.size == 0:
                n_units_by_angle[angle] = 0
                continue
            valid = ~np.isnan(mu).all(axis=(1, 2, 3))
            mu = mu[valid]
            local_ids = np.nonzero(valid)[0]
            n_units_by_angle[angle] = int(mu.shape[0])
            if mu.shape[0] == 0:
                continue
            muaps_list.append(mu.astype(np.float32))
            trial_labels.append(np.full(mu.shape[0], trial_idx, dtype=int))
            orig_unit_idx.append(local_ids.astype(int))

        if not muaps_list:
            continue
        muaps_all = np.concatenate(muaps_list, axis=0)
        trial_labels_arr = np.concatenate(trial_labels, axis=0)
        orig_unit_idx_arr = np.concatenate(orig_unit_idx, axis=0)
        trial_set = list(range(len(items)))

        print(f"  [tracking {task_name}] grid-search over angles {angles}")
        group_labels, group_sets, df_link_info, graph = assign_muaps_all_trials(
            muaps_all,
            trial_labels_arr,
            trial_set,
            assign_method="grid-search",
            dist_metric=DIST_METRIC,
            dist_thr=DIST_THR,
            sel_chs_method=SEL_CHS,
        )

        filtered_group_sets = []
        for chain in group_sets:
            hit_trials = {int(trial_labels_arr[u]) for u in chain}
            if len(hit_trials) >= 2:
                filtered_group_sets.append(chain)

        rows = []
        link_rows = []
        for group_id, chain in enumerate(filtered_group_sets, start=1):
            row = {
                "subject": subject,
                "location": location,
                "task": task_name,
                "group": group_id,
            }
            for angle in angles:
                row[f"{angle}deg"] = ""

            chain_sorted = sorted(chain, key=lambda u: angles[int(trial_labels_arr[u])])
            for u_global in chain_sorted:
                trial_idx = int(trial_labels_arr[u_global])
                angle = angles[trial_idx]
                row[f"{angle}deg"] = int(orig_unit_idx_arr[u_global])

            row["n_trials_hit"] = sum(row[f"{angle}deg"] != "" for angle in angles)
            rows.append(row)

            for u_from, u_to in zip(chain_sorted[:-1], chain_sorted[1:]):
                t_from = int(trial_labels_arr[u_from])
                t_to = int(trial_labels_arr[u_to])
                mu_from = np.expand_dims(muaps_all[u_from], axis=0)
                mu_to = np.expand_dims(muaps_all[u_to], axis=0)
                dist_mat, _ = compute_muaps_dist_sets(
                    mu_from,
                    mu_to,
                    dist_metric=DIST_METRIC,
                    sel_chs_method=SEL_CHS,
                )
                link_rows.append(
                    {
                        "subject": subject,
                        "location": location,
                        "task": task_name,
                        "group": group_id,
                        "from_angle": angles[t_from],
                        "to_angle": angles[t_to],
                        "from_unit_idx": int(orig_unit_idx_arr[u_from]),
                        "to_unit_idx": int(orig_unit_idx_arr[u_to]),
                        "NMSE": float(dist_mat[0, 0]),
                    }
                )

        df_tracks = pd.DataFrame(rows).sort_values("n_trials_hit", ascending=False) if rows else pd.DataFrame()
        df_links = pd.DataFrame(link_rows)

        table_dir = out_dir / "tracking" / "tables"
        fig_dir = out_dir / "tracking" / "figures"
        table_dir.mkdir(parents=True, exist_ok=True)
        fig_dir.mkdir(parents=True, exist_ok=True)

        if not df_tracks.empty:
            track_csv = table_dir / f"tracks_{task_name}_gridsearch_{DIST_METRIC}_thr{DIST_THR}.csv"
            save_dataframe(df_tracks, track_csv)
            save_dataframe(df_links, table_dir / f"tracking_links_{task_name}_gridsearch_{DIST_METRIC}_thr{DIST_THR}.csv")
            plot_tracking_arrows(
                df_tracks,
                angles,
                n_units_by_angle,
                fig_dir / f"tracking_arrows_{task_name}_gridsearch_{DIST_METRIC}_thr{DIST_THR}.png",
                f"{subject} {location} {task_name}: MU tracking",
            )

        if isinstance(df_link_info, pd.DataFrame) and not df_link_info.empty:
            df_link_info.to_csv(
                table_dir / f"raw_assign_link_info_{task_name}_gridsearch_{DIST_METRIC}_thr{DIST_THR}.csv",
                index=False,
                encoding="utf-8-sig",
            )

        if zero_trial_idx is not None:
            collect_tracking_validation(
                filtered_group_sets,
                muaps_all,
                trial_labels_arr,
                orig_unit_idx_arr,
                angles,
                zero_trial_idx,
                task_name,
                all_validation_f,
                all_validation_g,
            )
            collect_zero_anchor_adjacent_link_validation(
                filtered_group_sets,
                muaps_all,
                trial_labels_arr,
                angles,
                zero_trial_idx,
                task_name,
                all_validation_g,
            )

        all_tracks.append(df_tracks)
        all_link_rows.append(df_links)

    df_tracks_all = pd.concat(all_tracks, ignore_index=True) if all_tracks else pd.DataFrame()
    df_links_all = pd.concat(all_link_rows, ignore_index=True) if all_link_rows else pd.DataFrame()
    df_f = pd.DataFrame(all_validation_f)
    df_g = pd.DataFrame(all_validation_g)
    if not df_f.empty and not df_g.empty:
        plot_tracking_validation(df_f, df_g, out_dir / "tracking" / "figures" / f"tracking_validation_{DIST_METRIC}_thr{DIST_THR}.png")
        save_dataframe(df_f, out_dir / "tracking" / "tables" / "tracking_validation_nmse_by_angle.csv")
        save_dataframe(df_g, out_dir / "tracking" / "tables" / "tracking_validation_nmse_alternatives.csv")
    if not df_tracks_all.empty:
        save_dataframe(df_tracks_all, out_dir / "tracking" / "tables" / "all_tracks.csv")
    if not df_links_all.empty:
        save_dataframe(df_links_all, out_dir / "tracking" / "tables" / "all_tracking_links.csv")
    return df_tracks_all, df_links_all, pd.concat([df_f, df_g], ignore_index=True) if not df_f.empty or not df_g.empty else pd.DataFrame()


def parse_task_angle_from_name(name: str) -> tuple[str | None, int | None]:
    m_task = re.search(r"_flx_1_([^_]+)_15MVC_", name)
    m_angle = re.search(r"_decomp_(-?\d+)deg", name)
    task = m_task.group(1) if m_task else None
    angle = int(m_angle.group(1)) if m_angle else None
    return task, angle


## Module 9 — Tracking validation and alternative-link logic

Validates adjacent links in zero-anchored chains. `Second best` is the minimum finite NMSE among all non-linked MUs in the destination angle, while `Mean alternatives` is their finite mean.


In [ ]:
def collect_tracking_validation(
    filtered_group_sets,
    muaps_all: np.ndarray,
    trial_labels: np.ndarray,
    orig_unit_idx: np.ndarray,
    angles: list[int],
    zero_trial_idx: int,
    task_name: str,
    out_f: list[dict],
    out_g: list[dict],
):
    for chain in filtered_group_sets:
        zero_members = [u for u in chain if trial_labels[u] == zero_trial_idx]
        if not zero_members:
            continue
        u_zero = zero_members[0]
        muap_zero = np.expand_dims(muaps_all[u_zero], axis=0)
        for u_t in chain:
            trial_idx = int(trial_labels[u_t])
            angle = angles[trial_idx]
            idx_t = np.where(trial_labels == trial_idx)[0]
            muaps_t = muaps_all[idx_t]
            dist_mat, _ = compute_muaps_dist_sets(
                muap_zero,
                muaps_t,
                dist_metric=DIST_METRIC,
                sel_chs_method=SEL_CHS,
            )
            dist_array = dist_mat[0]
            local_ut = int(np.where(idx_t == u_t)[0][0])
            link_dist = float(dist_array[local_ut])
            out_f.append({"Task": task_name, "Angle": angle, "NMSE": link_dist})


def _parse_dist_list(value) -> np.ndarray:
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.asarray(value, dtype=float).reshape(-1)
        return arr[np.isfinite(arr)]
    if pd.isna(value):
        return np.asarray([], dtype=float)
    text = str(value).strip()
    if not text:
        return np.asarray([], dtype=float)
    text = text.strip("[]")
    if not text:
        return np.asarray([], dtype=float)
    parts = [p.strip() for p in text.replace("\n", " ").split(",")]
    vals = []
    for part in parts:
        try:
            vals.append(float(part))
        except ValueError:
            continue
    arr = np.asarray(vals, dtype=float)
    return arr[np.isfinite(arr)]


def collect_adjacent_link_validation(df_link_info: pd.DataFrame, task_name: str, out_g: list[dict]):
    required = {"label1", "label2", "link_dist", "no_link_dist"}
    if not required.issubset(df_link_info.columns):
        return
    for _, row in df_link_info.iterrows():
        try:
            label1 = int(row["label1"])
            label2 = int(row["label2"])
            link_dist = float(row["link_dist"])
        except Exception:
            continue
        if label2 - label1 != 1:
            continue
        alt_dists = _parse_dist_list(row["no_link_dist"])
        out_g.append({"Task": task_name, "Type": "Link", "NMSE": link_dist})
        if alt_dists.size:
            out_g.append({"Task": task_name, "Type": "Second best", "NMSE": float(np.min(alt_dists))})
            out_g.append({"Task": task_name, "Type": "Mean alternatives", "NMSE": float(np.mean(alt_dists))})


def collect_zero_anchor_adjacent_link_validation(
    filtered_group_sets,
    muaps_all: np.ndarray,
    trial_labels: np.ndarray,
    angles: list[int],
    zero_trial_idx: int,
    task_name: str,
    out_g: list[dict],
):
    for chain in filtered_group_sets:
        hit_trials = {int(trial_labels[u]) for u in chain}
        if zero_trial_idx not in hit_trials or len(hit_trials) < 2:
            continue
        chain_sorted = sorted(chain, key=lambda u: int(trial_labels[u]))
        for u_from, u_to in zip(chain_sorted[:-1], chain_sorted[1:]):
            t_from = int(trial_labels[u_from])
            t_to = int(trial_labels[u_to])
            if t_to - t_from != 1:
                continue
            idx_to_trial = np.where(trial_labels == t_to)[0]
            if idx_to_trial.size == 0:
                continue
            mu_from = np.expand_dims(muaps_all[u_from], axis=0)
            muaps_to = muaps_all[idx_to_trial]
            dist_mat, _ = compute_muaps_dist_sets(
                mu_from,
                muaps_to,
                dist_metric=DIST_METRIC,
                sel_chs_method=SEL_CHS,
            )
            target_pos = np.where(idx_to_trial == u_to)[0]
            if target_pos.size == 0:
                continue
            target_pos = int(target_pos[0])
            dist_array = dist_mat[0]
            link_dist = float(dist_array[target_pos])
            alt_dists = np.delete(dist_array, target_pos)
            alt_dists = alt_dists[np.isfinite(alt_dists)]
            out_g.append(
                {
                    "Task": task_name,
                    "Angle_pair": f"{angles[t_from]}->{angles[t_to]}",
                    "Type": "Link",
                    "NMSE": link_dist,
                }
            )
            if alt_dists.size:
                out_g.append(
                    {
                        "Task": task_name,
                        "Angle_pair": f"{angles[t_from]}->{angles[t_to]}",
                        "Type": "Second best",
                        "NMSE": float(np.min(alt_dists)),
                    }
                )
                out_g.append(
                    {
                        "Task": task_name,
                        "Angle_pair": f"{angles[t_from]}->{angles[t_to]}",
                        "Type": "Mean alternatives",
                        "NMSE": float(np.mean(alt_dists)),
                    }
                )


def plot_tracking_validation(df_f: pd.DataFrame, df_g: pd.DataFrame, out_path: Path):
    import seaborn as sns

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.2]})
    sns.lineplot(data=df_f, x="Angle", y="NMSE", errorbar="sd", ax=axes[0], color="#2c7fb8", linewidth=2)
    axes[0].set_ylabel("NMSE within MN group (n.u)")
    axes[0].set_xlabel("Angle (deg)")
    axes[0].set_title("f", loc="left", fontweight="bold")
    axes[0].set_ylim(bottom=-0.02)

    order = ["Link", "Second best", "Mean alternatives"]
    sns.boxplot(data=df_g, x="Type", y="NMSE", order=order, ax=axes[1], palette="crest", showfliers=False, width=0.6)
    sns.stripplot(data=df_g, x="Type", y="NMSE", order=order, ax=axes[1], color="skyblue", alpha=0.4, jitter=True, size=4)
    axes[1].axhline(y=DIST_THR, color="gray", linestyle="--", alpha=0.6)
    axes[1].set_ylabel("NMSE (n.u)")
    axes[1].set_xlabel("")
    axes[1].set_title("g", loc="left", fontweight="bold")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)


## Module 10 — Pipeline orchestration and reproducibility metadata

Runs MUAP extraction and tracking for all subjects and locations, combines global outputs, and records the analysis settings.


In [ ]:
def run_tracking(data_root: Path, out_root: Path):
    all_tracks = []
    all_links = []
    for subject, location, sub_dir in discover_subject_location_dirs(data_root):
        print(f"\n[tracking] {subject} {location}")
        out_dir = out_root / subject / location
        compute_muap_npz_for_dir(sub_dir, out_dir)
        tracks, links, _ = run_tracking_for_dir(out_dir, subject, location)
        if not tracks.empty:
            all_tracks.append(tracks)
        if not links.empty:
            all_links.append(links)

    global_dir = out_root / "_global" / "tables"
    if all_tracks:
        save_dataframe(pd.concat(all_tracks, ignore_index=True), global_dir / "all_tracking_tracks.csv")
    if all_links:
        save_dataframe(pd.concat(all_links, ignore_index=True), global_dir / "all_tracking_links.csv")


def write_run_info(out_root: Path, args):
    out_root.mkdir(parents=True, exist_ok=True)
    info = {
        "data_root": str(args.data_root),
        "out_root": str(args.out_root),
        "match_roa_threshold": args.match_roa_thr,
        "tol_spike_ms": TOL_SPIKE_MS,
        "tol_train_ms": TOL_TRAIN_MS,
        "silhouette_logic": "MATLAB calcSIL peak-detection + 2-means on pulse-train peaks",
        "tracking_metric": DIST_METRIC,
        "tracking_threshold": DIST_THR,
        "tracking_selected_channels": SEL_CHS,
        "muap_window_ms": WIN_MS,
    }
    with open(out_root / "run_info.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(info.keys()))
        writer.writeheader()
        writer.writerow(info)
    try:
        shutil.copy2(Path(__file__), out_root / "run_report_analysis_all.py")
    except Exception:
        pass


## Module 11 — Run the analysis

Choose which pipeline branches to run. Both switches reproduce the default behavior of the latest Python script.


In [ ]:
# Set either switch to False when only one branch needs to be regenerated.
RUN_METRICS = True
RUN_TRACKING = True

notebook_args = argparse.Namespace(
    data_root=DATA_ROOT,
    out_root=RESULT_ROOT,
    match_roa_thr=MATCH_ROA_THR,
)

RESULT_ROOT.mkdir(parents=True, exist_ok=True)
write_run_info(RESULT_ROOT, notebook_args)

print("DATA ROOT:", DATA_ROOT)
print("OUT ROOT :", RESULT_ROOT)
print("MATCH ROA THR:", MATCH_ROA_THR)

if RUN_METRICS:
    run_metrics(DATA_ROOT, RESULT_ROOT, MATCH_ROA_THR)
if RUN_TRACKING:
    run_tracking(DATA_ROOT, RESULT_ROOT)

print("\nDone.")
